# Deformation of solids (2D plate with holes)

**Prashant K. Jha**

*Mechanical Engineering, South Dakota School of Mines and Technology, Rapid City, SD, USA*

In this tutorial, we model the small-strain deformation of an elastic 3D body using FEniCSx.
 
## Problem description

We consider a 2-D domain $\Omega$ bounded by rectangle:
$$
\{(x,y): x \in (0,0.8),\; y \in (0,1.2)\},
$$
We consider a triangle mesh. Meshing is done using `gmsh` library. We consider a plane stress approximation with unit meter thickness in $z$ direction.

<img src="mesh_2d/mesh_h_0.020.png" style="width:600px;">

## Linear elastic theory

We apply the same theory from 3-D problem but now for 2-D domain. Here displacement $\boldsymbol{u}(\boldsymbol{X})$ is a 2-D vector function. 

### Geometrical, material, and external loading parameters and functions

We assume no body force, i.e., $\boldsymbol{f} = \boldsymbol{0}$.

The displacement is fully constrained on the bottom face ($y = 0$ face)
$$
\Gamma_D = \{(x,y) \in \partial\Omega : y = 0\},
$$
i.e., $\boldsymbol{u} = \boldsymbol{0}$ on $\Gamma_D$.

A constant traction load is applied on the top face ($y = 1.2$ face)
$$
\Gamma_N = \{(x,y) \in \partial\Omega : y = 1.2\},
$$
of the form
$$
\boldsymbol{t} = (t_{x0}, t_{y0}),
$$
where $t_{x0}$ and $t_{y0}$ are scalar load parameters (force per unit area) that can be varied, e.g. for fatigue-type loading studies.

As for the material properties, we take Young's modulus $E = 200$ GPa and Poisson ratio $\nu = 0.3$. Given $(E, \nu)$, Lam\`e parameters can be determined using:
\begin{equation}
\lambda = \frac{E\nu}{(1+\nu)(1-2\nu)}, \qquad \mu = \frac{E}{2(1+\nu)}.
\end{equation}

# Results

We take $t_{x0} = 10^5$ N/$\text{m}^2$ and $t_{y0} = 10^7$ N/$\text{m}^2$. In visualization below, we scaled the displacement by the factor 500 to highlight the deformation.

<img src="fwd_result/linear_elastic_mesh_2d/result.png" style="width:600px;">

### Fenics implementation

We start by loading the relevant packages:

In [1]:
import dolfinx
import numpy as np
import sys
import os

from mpi4py import MPI
comm = MPI.COMM_WORLD
rank = comm.Get_rank()

from petsc4py import PETSc

# specific functions from dolfinx modules
from dolfinx import fem, mesh, io, plot, log
from dolfinx.fem import (Constant, dirichletbc, Function, functionspace, Expression )
from dolfinx.fem.petsc import LinearProblem
from dolfinx.io import VTXWriter


# specific functions from ufl modules
import ufl
from ufl import (TestFunctions,TestFunction, TrialFunction, Identity, grad, det, div, dev, inv, tr, sqrt, conditional ,\
                 gt, dx, inner, derivative, dot, ln, split, outer, cos, acos, lt, eq, ge, le, exp)

# basix finite elements
import basix
from basix.ufl import element, mixed_element, quadrature_element

# for plotting
import matplotlib.pyplot as plt
plt.close('all')

from datetime import datetime

log.set_log_level(log.LogLevel.WARNING)

Next, we set values of different parameters in the simulation:

In [2]:
# geometry
with io.XDMFFile(MPI.COMM_WORLD, "mesh_2d/mesh_h_0.020.xdmf", "r") as xdmf:
    domain = xdmf.read_mesh(name="Grid")

# infer bounds from mesh
# Some 2D meshes are embedded in 3D coordinates (x, y, z), so use only x and y.
coords = domain.geometry.x
x_min = np.min(coords[:, 0])
y_min = np.min(coords[:, 1])
x_max = np.max(coords[:, 0])
y_max = np.max(coords[:, 1])
omega_L = x_max - x_min
omega_W = y_max - y_min

# get nodal coordinates in reference configuration
x = ufl.SpatialCoordinate(domain) 

# material
E = Constant(domain, PETSc.ScalarType(2.e+11)) #200 GPa
nu = Constant(domain, PETSc.ScalarType(0.3))
lamda = E*nu/(1+nu)/(1-2*nu)
mu = E/(2*(1+nu))

# loading
# Traction applied on the top edge (y = y_max): t = (tx0, ty0)
# Set these parameters before running.
tx0_value = 1e+5
ty0_value = 1e+7
tx0 = Constant(domain, PETSc.ScalarType(tx0_value))
ty0 = Constant(domain, PETSc.ScalarType(ty0_value))
t_scale = Constant(domain, PETSc.ScalarType(0.0))  # allows write_sim(0.0) before the solve

Define vector finite element function space and scalar finite element space (scalar function space will be used to compute the post-processing quantities such as magnitude of the displacement and von-Mises stress):

In [3]:
# specify order of interpolation
p_order = 1

# create FE element (vector) for 2D displacement: u = (ux, uy)
Uvec = element("Lagrange", domain.basix_cell(), p_order, shape=(2,))

# FE function space (vector)
Vvec = fem.functionspace(domain, Uvec)

# space for von Mises stress (scalar)
U = element("Lagrange", domain.basix_cell(), p_order)
V = fem.functionspace(domain, U)

We also define the trial and test function, and infer the spatial dimension of the problem:

In [4]:
u_trial = TrialFunction(Vvec)
v = TestFunction(Vvec)

# spatial dimension of the problem
d = len(u_trial)
print('d = ', d)

d =  2


Next, define boundaries for boundary conditions:

In [5]:
# define edges for BCs on a 2D mesh (x-y plane)
def yBot(x):
    return np.isclose(x[1], y_min)  # bottom edge (y = 0)
def yTop(x):
    return np.isclose(x[1], y_max)  # top edge (y = 1.2)

# mark the sub-domains
boundaries = [(1, yBot), (2, yTop)]

# build collections of facets on each subdomain and mark them appropriately.
facet_indices, facet_markers = [], [] 
fdim = domain.topology.dim - 1
for (marker, locator) in boundaries:
    facets = mesh.locate_entities(domain, fdim, locator) 
    facet_indices.append(facets) 
    facet_markers.append(np.full_like(facets, marker)) 

# Format the facet indices and markers as required for use in dolfinx.
facet_indices = np.hstack(facet_indices).astype(np.int32)
facet_markers = np.hstack(facet_markers).astype(np.int32)
sorted_facets = np.argsort(facet_indices)
 
# Add these marked facets as "mesh tags" for later use in BCs.
facet_tags = mesh.meshtags(domain, fdim, facet_indices[sorted_facets], facet_markers[sorted_facets])

# create connectivity between the 2D and 3D entities
domain.topology.create_connectivity(domain.topology.dim-1, domain.topology.dim)

Define the surface area measure for integration over the right boundary for the traction boundary condition:

In [6]:
# dx for integration over the domain
# dx = ufl.Measure('dx', domain=domain, metadata={'quadrature_degree': 2})

# ds for integration over the surface
ds = ufl.Measure('ds', domain=domain, subdomain_data=facet_tags, metadata={'quadrature_degree':2})

Implement Dirichlet boundary condition on displacement next:

In [7]:
# Bottom edge (y = y_min) displacement degrees of freedom
Btm_dofs_u1 = fem.locate_dofs_topological(Vvec.sub(0), facet_tags.dim, facet_tags.find(1))
Btm_dofs_u2 = fem.locate_dofs_topological(Vvec.sub(1), facet_tags.dim, facet_tags.find(1))

# Dirichlet BCs: fully fixed (ux = uy = 0) on y = 0
bcs_0 = dirichletbc(0.0, Btm_dofs_u1, Vvec.sub(0))
bcs_1 = dirichletbc(0.0, Btm_dofs_u2, Vvec.sub(1))

bcs = [bcs_0, bcs_1]

Next, we define the body force and traction:

In [8]:
# body force
f = Constant(domain, PETSc.ScalarType((0.0, 0.0)))#, PETSc.ScalarType)

# traction on the top edge (y = y_max)
# t = (tx0, ty0), scaled by t_scale so we can output t=0 (initial state) and then solve for t=1
t = ufl.as_vector((t_scale * tx0, t_scale * ty0))

Finally, we are ready to define the bilinear and linear forms associated with the boundary value problem on displacement:

In [9]:
# Define strain and stress
def epsilon(u):
    return 0.5*(grad(u) + grad(u).T)
    #return sym(nabla_grad(u))

def sigma(u):
    return lamda*div(u)*Identity(d) + 2*mu*epsilon(u)

# bilinear form
a = inner(sigma(u_trial), epsilon(v))*dx

# linear form
L = inner(f, v)*dx + inner(t, v)*ds(2)

In [10]:
u = Function(Vvec, name = "u")

problem = LinearProblem(a, L, bcs=bcs, u = u, petsc_options={"ksp_type": "preonly", "pc_type": "lu"})
# problem.solve()

Compute post-processing quantities such as von-Mises stress that is defined as
\begin{equation}
\sigma_v = \sqrt{\frac{3}{2}\boldsymbol{\sigma}_{dev}\boldsymbol{\colon}\boldsymbol{\sigma}_{dev}} = \sqrt{\frac{3}{2}\sigma_{dev, ij} \sigma_{dev, ij}}\;, \quad \text{where}\quad  \boldsymbol{\sigma}_{dev} = \text{deviatoric stress} = \boldsymbol{\sigma} - \frac{\mathrm{tr}(\boldsymbol{\sigma})}{3}\boldsymbol{I} 
\end{equation}
and magnitude of the displacement.

In [11]:
# Post-processing: compute von Mises stress
sigma_dev = sigma(u) - (1./3)*tr(sigma(u))*Identity(d)  # deviatoric stress
vm = sqrt((3/2)*inner(sigma_dev, sigma_dev))
sigma_vm_expr = Expression(vm, V.element.interpolation_points())
sigma_vm = Function(V, name = 'von Mises stress')
sigma_vm.interpolate(sigma_vm_expr)

# Compute magnitude of displacement
u_magnitude_expr = Expression(sqrt(dot(u, u)), V.element.interpolation_points())
u_magnitude = Function(V, name = 'magnitude(u)')
u_magnitude.interpolate(u_magnitude_expr)

u_mag_max = u_magnitude.x.array.max()
u_mag_min = u_magnitude.x.array.min()
sigma_vm_max = sigma_vm.x.array.max()

print('min/max u:', u_mag_min, u_mag_max)
print('max von Mises stress:', sigma_vm_max)


# output results
results_folder = "fwd_result/linear_elastic_mesh_2d"
os.makedirs(results_folder, exist_ok=True)
filename = results_folder + "/solution"
file_results = io.VTXWriter(domain.comm, filename + ".bp", [u, sigma_vm, u_magnitude], engine="BP4")

def write_sim(t):
    sigma_vm.interpolate(sigma_vm_expr)
    u_magnitude.interpolate(u_magnitude_expr)
    file_results.write(t)

    u_mag_max = u_magnitude.x.array.max()
    u_mag_min = u_magnitude.x.array.min()
    sigma_vm_max = sigma_vm.x.array.max()
    print('min/max u:', u_mag_min, u_mag_max)
    print('max von Mises stress:', sigma_vm_max)

min/max u: 0.0 0.0
max von Mises stress: 0.0


We now solve the problem using fenics in-build solver function:

In [12]:
log.set_log_level(log.LogLevel.INFO)

write_sim(0.0)

## Solve for fixed traction load
u.x.array[:] = 0.0
t_scale.value = 1.0


problem.solve()
write_sim(1.0)

min/max u: 0.0 0.0
max von Mises stress: 0.0
min/max u: 0.0 0.00010447147622710773
max von Mises stress: 47823364.146307


### Plotting results in paraview
To plot, open a paraview app and drag the folder `solution.bp` in the box `Pipieline Browser` on left side of paraview app. Next, select the `Warp by Vector` from the `Filters` menu to add displacement to the reference configuration to get the current configuration. Next, select the `von Mises stress` from the plot field selector. 

<img src="fwd_result/linear_elastic_mesh_2d/result.png" style="width:600px;">
